# Synthetic Data Generator for product reviews


In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 gradio

In [ ]:
import json
import random
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if 'T4' in gpu_info:
    print("Connected to T4 GPU")
else:
    print("WARNING: No T4 detected. Check your runtime.")

In [ ]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
QWEN = "Qwen/Qwen2.5-3B-Instruct"

MODELS = {
    "LLaMA 3.2 3B": LLAMA,
    "Qwen 2.5 3B": QWEN,
}

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
PRODUCTS = [
    "Solar-Powered Flashlight",
    "Bluetooth-Enabled Toaster",
    "AI-Powered Houseplant Monitor",
    "Self-Stirring Coffee Mug",
    "Noise-Cancelling Pillow",
    "Smart Yoga Mat",
    "Holographic Business Cards",
    "DNA-Based Perfume Generator",
]

SYSTEM_PROMPTS = {
    "positive": (
        "You are an enthusiastic customer who loves quirky gadgets. "
        "You write short, genuine-sounding product reviews (2-3 sentences). "
        "Always include a star rating on the first line like 'Rating: 5/5'. "
        "Then write the review. Nothing else."
    ),
    "negative": (
        "You are a disappointed customer who regrets buying a gadget. "
        "You write short, critical product reviews (2-3 sentences). "
        "Always include a star rating on the first line like 'Rating: 1/5'. "
        "Then write the review. Nothing else."
    ),
    "neutral": (
        "You are a balanced, fair customer writing an honest review. "
        "You write short, measured product reviews (2-3 sentences) mentioning one pro and one con. "
        "Always include a star rating on the first line like 'Rating: 3/5'. "
        "Then write the review. Nothing else."
    ),
}

USER_PROMPTS = {
    "positive": "Write a glowing review for this product: {product}",
    "negative": "Write a disappointed review for this product: {product}",
    "neutral": "Write a balanced, honest review for this product: {product}",
}

In [ ]:
def load_model(model_name):
    """Load a quantized model and its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config
    )
    return tokenizer, model


def unload_model(tokenizer, model):
    """Free GPU memory so we can load the next model."""
    del model
    del tokenizer
    torch.cuda.empty_cache()


def generate_review(tokenizer, model, product, sentiment):
    """Generate a single synthetic review."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[sentiment]},
        {"role": "user", "content": USER_PROMPTS[sentiment].format(product=product)},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=150,
        temperature=0.8,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode ONLY the new tokens (skip the input prompt)
    new_tokens = outputs[0][inputs.shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

In [ ]:
def generate_dataset(selected_models, selected_products, selected_sentiments, reviews_per_combo):
    """
    Main pipeline: loops through every combination of
    model × product × sentiment and generates reviews.
    Loads one model at a time to fit in T4 memory.
    """
    reviews_per_combo = int(reviews_per_combo)
    dataset = []
    log = ""

    for model_label in selected_models:
        model_name = MODELS[model_label]
        log += f"\n⏳ Loading {model_label}...\n"
        yield dataset, log  # stream progress to the UI

        tokenizer, model = load_model(model_name)
        log += f" {model_label} loaded!\n"
        yield dataset, log

        for product in selected_products:
            for sentiment in selected_sentiments:
                for i in range(reviews_per_combo):
                    log += f"  Generating: {product} | {sentiment} | #{i+1}...\n"
                    yield dataset, log

                    review_text = generate_review(tokenizer, model, product, sentiment)

                    dataset.append({
                        "product": product,
                        "sentiment": sentiment,
                        "review": review_text,
                        "model": model_label,
                    })

                    yield dataset, log

        log += f" Unloading {model_label} to free GPU memory...\n"
        unload_model(tokenizer, model)
        yield dataset, log

    log += f"\n Done! Generated {len(dataset)} reviews.\n"
    yield dataset, log

In [ ]:
with gr.Blocks(title="Absurd Product Review Generator") as ui:

    gr.Markdown("# Synthetic Product Review Generator")
    gr.Markdown(
        "Generate diverse, labeled training data using multiple open-source LLMs. "
        "Pick your models, products, sentiments, and hit Generate!"
    )

    with gr.Row():
        with gr.Column():
            model_picker = gr.CheckboxGroup(
                choices=list(MODELS.keys()),
                value=list(MODELS.keys()),
                label="Models to use",
            )
            product_picker = gr.CheckboxGroup(
                choices=PRODUCTS,
                value=PRODUCTS[:3],
                label="Products to review",
            )
            sentiment_picker = gr.CheckboxGroup(
                choices=["positive", "negative", "neutral"],
                value=["positive", "negative", "neutral"],
                label="Sentiments",
            )
            num_slider = gr.Slider(
                minimum=1, maximum=3, value=1, step=1,
                label="Reviews per combination",
            )
            generate_btn = gr.Button("Generate Dataset", variant="primary")

        with gr.Column():
            log_output = gr.Textbox(label="Progress Log", lines=15, interactive=False)
            data_output = gr.JSON(label="Generated Dataset")
            file_output = gr.File(label=" Download JSON", visible=False)

    def run_and_save(models, products, sentiments, num):
        result = []
        log = ""
        for dataset, log in generate_dataset(models, products, sentiments, num):
            result = dataset
            yield result, log, gr.File(visible=False)

        filepath = "/content/synthetic_reviews.json"
        with open(filepath, "w") as f:
            json.dump(result, f, indent=2)

        yield result, log + "\n File saved! Click download below.", gr.File(value=filepath, visible=True)

    generate_btn.click(
        fn=run_and_save,
        inputs=[model_picker, product_picker, sentiment_picker, num_slider],
        outputs=[data_output, log_output, file_output],
    )

ui.launch(debug=True, theme=gr.themes.Soft())

GOOGLE COLAB: https://colab.research.google.com/drive/1bvtroWS84FVwgnca7FulWE2zxhj1HHeD#scrollTo=QJ1qpySJTtGy